# Data Quality com Soda Core

Este notebook realiza a validação de qualidade de dados (Data Quality) utilizando a biblioteca **Soda Core**. As validações abrangem completude, unicidade, validade de formatos e integridade referencial entre as tabelas de clientes, produtos e vendas.

In [41]:
import pandas as pd
import json
from soda.scan import Scan
from datetime import datetime
import duckdb

print("="*80)
print("DATA QUALITY COM SODA CORE")
print("="*80)
print(f"\nIniciado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

DATA QUALITY COM SODA CORE

Iniciado em: 2026-01-28 00:54:12


In [44]:
# ===== CARREGAR DADOS RAW =====
print("\nCarregando dados para validação...")

customers_path = '../data/raw/customers_raw.csv'
products_path = '../data/raw/products_raw.json'
sales_path = '../data/raw/sales_raw.csv'

try:
    df_customers = pd.read_csv(customers_path)
    with open(products_path, 'r') as f:
        df_products = pd.DataFrame(json.load(f))
    df_sales = pd.read_csv(sales_path)
    
    print(f"- Customers: {len(df_customers):,} registros")
    print(f"- Products: {len(df_products):,} registros")
    print(f"- Sales: {len(df_sales):,} registros")
except Exception as e:
    print(f"Erro ao carregar arquivos: {e}")

# Definindo coluna margin para validação de margem negativa em products
df_products["margin"] = df_products["list_price"] - df_products["unit_cost"]



Carregando dados para validação...
- Customers: 5,000 registros
- Products: 10,000 registros
- Sales: 120,000 registros


In [ ]:
con = duckdb.connect()

con.register("customers", df_customers)
con.register("products", df_products)
con.register("sales", df_sales)

scan = Scan()
scan.set_data_source_name("duckdb")
scan.add_duckdb_connection(con)

# Definir as verificações (SodaCL)
checks_yaml = """
checks for customers:
  # Existência
  - row_count > 0

  # Chave primária
  - missing_count(customer_id) = 0
  - duplicate_count(customer_id) = 0

  # Completude (campos críticos)
  - missing_count(company_name) = 0

checks for products:
  - row_count > 0

  # Chave primária
  - missing_count(product_id) = 0
  - duplicate_count(product_id) = 0

  # Regras numéricas
  - min(unit_cost) >= 0
  - min(list_price) >= 0
  
  # Regras de negócio
  - failed rows:
      name: Produtos com margem negativa
      fail condition: margin <= 0

checks for sales:
  - row_count > 0

  # Chave primária
  - missing_count(sale_id) = 0
  - duplicate_count(sale_id) = 0

  # Integridade referencial
  - values in (customer_id) must exist in customers (customer_id)
  - values in (product_id) must exist in products (product_id)

  # Regras de negócio
  - min(quantity) >= 1
  - min(total_price) >= 0
"""

scan.add_sodacl_yaml_str(checks_yaml)
scan.execute()

INFO:soda.scan:[00:56:01] Soda Core 3.5.6
INFO:soda.scan:[00:56:02] Using DefaultSampler
INFO:soda.scan:[00:56:02] Scan summary:
INFO:soda.scan:[00:56:02] 16/17 checks PASSED: 
INFO:soda.scan:[00:56:02]     customers in duckdb
INFO:soda.scan:[00:56:02]       row_count > 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]       missing_count(customer_id) = 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]       duplicate_count(customer_id) = 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]       missing_count(company_name) = 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]     products in duckdb
INFO:soda.scan:[00:56:02]       row_count > 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]       missing_count(product_id) = 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]       duplicate_count(product_id) = 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]       min(unit_cost) >= 0 [sodacl_string.yml] [PASSED]
INFO:soda.scan:[00:56:02]     

2

## Conclusão da Auditoria

### TABELA CUSTOMERS
As validações aplicadas à tabela customers garantem a integridade básica e a completude dos dados de clientes:

- Existência de dados:
    - Verificação de que a tabela possui registros (row_count > 0)
- Integridade da chave primária:
    - Ausência de valores nulos em customer_id
    - Unicidade do customer_id (sem registros duplicados)
- Completude de atributo crítico:
    - Ausência de valores nulos no campo company_name

### TABELA PRODUCTS
Para a tabela products, foram realizadas validações focadas em integridade e regras numéricas de negócio:

- Existência de dados:
    - Garantia de que a tabela contém registros
- Integridade da chave primária:
    - product_id sem valores nulos
    - product_id sem duplicidades
- Regras de negócio:
    - 1 PROBLEMA IDENTIFICADO: 359 linhas com margem negativa (list_price > unit_cost)

### TABELA SALES
A tabela sales, por ser transacional e central para análises, recebeu o maior conjunto de validações:

- Existência de dados:
    - Confirmação de que a tabela possui registros
- Integridade da chave primária:
    - sale_id sem valores nulos
    - sale_id sem registros duplicados
- Integridade referencial:
    - Todos os customer_id existentes em sales possuem correspondência na tabela customers
    - Todos os product_id existentes em sales possuem correspondência na tabela products
- Regras de negócio:
    - Quantidade mínima por venda (quantity >= 1)
    - Valor total da venda não negativo (total_price >= 0)


## Resultado Final do Scan

- 16 de 17 checks aprovados
- 1 falha detectada, relacionada à regra de negócio de margem negativa:
   - A validação identificou 359 registros com margem negativa (list_price < unit_cost)
- Nenhum alerta gerado
- Nenhum erro de execução

Conclusão:
A maioria dos critérios de qualidade definidos foi atendida com sucesso.
A única falha identificada refere-se a uma regra de negócio.

Após a correção desses registros e reexecução do scan, espera-se que o conjunto de dados atinja 100% de conformidade, tornando-se plenamente apto para uso em análises, relatórios e processos analíticos subsequentes. Será corrigido no notebook "03_data_trasformation.ipynb".